# Importing Libraries

In [80]:
import pandas as pd
import numpy as np
import re

## Utils

In [83]:
import numpy as np
import pandas as pd

def _normalize_epoch_to_seconds(s: pd.Series) -> pd.Series:
    """
    Make frame.time_epoch consistent (float seconds).
    Handles mixed units: seconds / ms / µs / ns.
    Drops inf/NaN and maps extreme outliers to NaN for later drop.
    """
    # Coerce to numeric
    x = pd.to_numeric(s, errors='coerce')
    # Remove ±inf
    x[~np.isfinite(x)] = np.nan

    # Heuristic unit detection:
    # < 1e11  -> seconds (up to ~5138-11-16 if seconds; typical UNIX seconds now ~1.7e9)
    # [1e11, 1e14)  -> milliseconds
    # [1e14, 1e17)  -> microseconds
    # >= 1e17 -> nanoseconds
    sec_mask = x < 1e11
    ms_mask  = (x >= 1e11) & (x < 1e14)
    us_mask  = (x >= 1e14) & (x < 1e17)
    ns_mask  = (x >= 1e17)

    x_sec = x.copy()
    x_sec[ms_mask] = x[ms_mask] / 1e3
    x_sec[us_mask] = x[us_mask] / 1e6
    x_sec[ns_mask] = x[ns_mask] / 1e9

    # After scaling, anything wildly out of plausible UNIX time is set NaN
    # (valid UNIX seconds range roughly [0, ~2e10] for our use)
    x_sec[(x_sec < 0) | (x_sec > 2e10)] = np.nan
    return x_sec

def add_timestamp_column(df: pd.DataFrame, col='frame.time_epoch', ts_col='ts') -> pd.DataFrame:
    df = df.copy()
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

    # Normalize to float seconds
    df[col] = _normalize_epoch_to_seconds(df[col])
    # Drop rows we cannot parse
    before = len(df)
    df = df.dropna(subset=[col])
    after = len(df)
    if before != after:
        print(f"[timestamp cleanup] Dropped {before - after} rows with invalid timestamps.")

    # Build timezone-naive pandas datetime (ns) from seconds
    df[ts_col] = pd.to_datetime(df[col], unit='s', errors='coerce')
    # Final safety drop if any still failed
    before2 = len(df)
    df = df.dropna(subset=[ts_col])
    if before2 != after:
        print(f"[timestamp cleanup] Dropped additional {before2 - after} rows failing to convert to datetime.")

    return df


## Dataset

In [86]:
dataset = "CIC"
df = pd.read_csv(f"{dataset}/combined/combined_raw.csv")

/var/folders/9h/68tfr4bn46xgf89px2064q7c0000gn/T/ipykernel_85202/3106155759.py:2: DtypeWarning: Columns (17,18,19,20,21,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{dataset}/combined/combined_raw.csv")


In [87]:
df["label"].unique()

array([nan, 'Amcrest Camera', 'Arlo Q Camera', 'Atomi Coffee Maker',
       'Borun Camera', 'Eufy HomeBase 2', 'Globe Lamp',
       'Gosund Plug - Roomba', 'HeimVision Camera', 'Luohe Camera',
       'Netatmo Camera', 'Philips Hue Bridge', 'Ring Base Station',
       'SimCam'], dtype=object)

In [88]:
df.count()

frame.number        24667494
frame.time_epoch    24667494
frame.len           24667494
eth.src             24667494
eth.dst             24667494
eth.type            24629079
ip.src              23775881
ip.dst              23775881
ip.proto            23673034
tcp.srcport          7342244
tcp.dstport          7342244
tcp.stream           7342244
tcp.window_size      7342244
tcp.len              7342244
udp.srcport         15884494
udp.dstport         15884494
udp.stream          15884494
dns.qry.name          144752
http                  105796
ntp                    56174
label               15037522
label_norm          15037522
label_canon         24667494
label_canon_norm    24667494
attack_flag         24667494
attack_family       15037522
attack_tool         15037522
transport           15037522
dtype: int64

In [89]:
df["attack_flag"].value_counts()

attack_flag
1    15037522
0     9629972
Name: count, dtype: int64

In [90]:
df["label"].value_counts()

label
Philips Hue Bridge      4970524
Netatmo Camera          3288873
Atomi Coffee Maker      3006090
Arlo Q Camera           2428326
Eufy HomeBase 2          603930
SimCam                   386428
HeimVision Camera        101620
Globe Lamp                76548
Gosund Plug - Roomba      59324
Amcrest Camera            46867
Ring Base Station         41394
Luohe Camera              27347
Borun Camera                251
Name: count, dtype: int64

In [ ]:
df["label"]

In [96]:
df.head()

,frame.number,frame.time_epoch,frame.len,eth.src,eth.dst,eth.type,ip.src,ip.dst,ip.proto,tcp.srcport,...,http,ntp,label,label_norm,label_canon,label_canon_norm,attack_flag,attack_family,attack_tool,transport
0,88,1.635852e+09,60,9c:8e:cd:1d:ab:9f,3c:18:a0:41:c3:a0,0x0806,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,AMCREST WiFi Camera,amcrest wifi camera,0,NaN,NaN,NaN
1,94,1.635852e+09,60,9c:8e:cd:1d:ab:9f,ff:ff:ff:ff:ff:ff,0x0806,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,AMCREST WiFi Camera,amcrest wifi camera,0,NaN,NaN,NaN
2,1922,1.635852e+09,95,9c:8e:cd:1d:ab:9f,3c:18:a0:41:c3:a0,0x0800,192.168.137.106,54.205.158.199,17.0,NaN,...,NaN,NaN,NaN,NaN,AMCREST WiFi Camera,amcrest wifi camera,0,NaN,NaN,NaN
3,3907,1.635852e+09,95,9c:8e:cd:1d:ab:9f,3c:18:a0:41:c3:a0,0x0800,192.168.137.106,54.205.158.199,17.0,NaN,...,NaN,NaN,NaN,NaN,AMCREST WiFi Camera,amcrest wifi camera,0,NaN,NaN,NaN
4,3988,1.635852e+09,74,9c:8e:cd:1d:ab:9f,3c:18:a0:41:c3:a0,0x0800,192.168.137.106,192.168.137.1,6.0,44397.0,...,NaN,NaN,NaN,NaN,AMCREST WiFi Camera,amcrest wifi camera,0,NaN,NaN,NaN


In [98]:
df["attack_family"].unique()

array([nan, '2-RTSP Brute Force', '1-Flood'], dtype=object)

In [100]:
df["attack_tool"].unique()

array([nan, 'Hydra', 'Nmap', 'Unknown'], dtype=object)

In [102]:
df["transport"].unique()

array([nan, 'Unknown', 'TCP', 'HTTP', 'UDP'], dtype=object)

## Temporary Subset of Data

In [52]:
label_to_drop = "Arlo Q Camera"

In [54]:
df = df.drop(df[df['label'] == label_to_drop].index)

In [74]:
df.dropna(subset=['label'], inplace=True)

## Performing Feature Engineering

In [13]:
import numpy as np
import pandas as pd

# ===========
# 1) Utilities
# ===========

def shannon_entropy(series: pd.Series) -> float:
    vals = series.dropna().astype(str)
    if len(vals) == 0:
        return 0.0
    p = vals.value_counts(normalize=True).values
    return float(-(p * np.log2(p)).sum())

def ensure_cols(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def detect_epoch_unit(x: pd.Series) -> str:
    """
    Detects likely unit of an epoch column: 's', 'ms', or 'us'.
    Uses median value heuristic.
    """
    s = pd.to_numeric(x, errors='coerce').dropna()
    if s.empty:
        return 's'
    m = s.median()
    # Year 2001-2050 ~ seconds range [~1e9, ~2.5e9]
    if m > 1e14:   # definitely microseconds
        return 'us'
    if m > 1e12:   # likely milliseconds in the trillions
        return 'ms'
    return 's'

def add_timestamp_column(df: pd.DataFrame, col='frame.time_epoch', ts_col='ts') -> pd.DataFrame:
    """
    Create a pandas datetime column from an epoch column with unknown units.
    Handles seconds / milliseconds / microseconds.
    Drops rows with non-coercible or out-of-bounds timestamps.
    """
    df = df.copy()
    ensure_cols(df, [col])

    # Coerce numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=[col])

    unit = detect_epoch_unit(df[col])
    if unit == 's':
        factor = 1.0
    elif unit == 'ms':
        factor = 1e-3
    else:
        factor = 1e-6

    # Convert to seconds float then to datetime
    sec = df[col] * factor
    # clip to avoid out-of-bounds datetimes
    # (pandas supports roughly year 1677..2262)
    # we'll clip to a safe range of [2000-01-01, 2100-01-01]
    lower = pd.Timestamp('2000-01-01').timestamp()
    upper = pd.Timestamp('2100-01-01').timestamp()
    sec = sec.clip(lower=lower, upper=upper)

    df[ts_col] = pd.to_datetime(sec, unit='s', utc=True)
    return df

def safe_str(x):
    return str(x) if pd.notna(x) else 'NA'

def safe_pair(a, b):
    """
    Return an unordered, hashable pair with 'NA' placeholders.
    """
    A, B = safe_str(a), safe_str(b)
    return tuple(sorted((A, B)))

In [14]:


# ============================
# 2) Individual feature steps
# ============================

def step_00_prepare(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic hygiene: ensure key columns exist, create ts, sort.
    Also ensure 'attack_flag' exists.
    """
    must = ['frame.time_epoch', 'frame.len', 'ip.src', 'ip.dst', 'label']
    ensure_cols(df, must)
    if 'attack_flag' not in df.columns:
        df = df.copy()
        df['attack_flag'] = 0

    df = add_timestamp_column(df, col='frame.time_epoch', ts_col='ts')
    df = df.sort_values(['label', 'ts']).reset_index(drop=True)
    return df

def step_01_window_bin(df: pd.DataFrame, win_seconds: int = 60) -> pd.DataFrame:
    """
    Add absolute time bins (floor to win_seconds).
    """
    df = df.copy()
    # Use timestamp in seconds since epoch to bin
    df['time_bin'] = (df['ts'].astype('int64') // 10**9) // win_seconds
    df['time_bin'] = df['time_bin'].astype('int64')
    return df

def step_02_inter_packet_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inter-packet time per device.
    """
    df = df.copy()
    df['inter_packet_time'] = (
        df.groupby('label')['ts']
          .diff()
          .dt.total_seconds()
          .fillna(0.0)
          .clip(lower=0.0)
    )
    return df

def step_03_ports_defaults(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure port columns exist (fill with NaN if missing).
    """
    df = df.copy()
    for c in ['tcp.srcport','tcp.dstport','udp.srcport','udp.dstport']:
        if c not in df.columns:
            df[c] = np.nan
    return df

def step_04_pairs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # pre-cast to strings with NA -> 'NA'
    t_s = df['tcp.srcport'].astype('string')
    t_d = df['tcp.dstport'].astype('string')
    u_s = df['udp.srcport'].astype('string')
    u_d = df['udp.dstport'].astype('string')

    tcp_pair = [safe_pair(a, b) for a, b in zip(t_s, t_d)]
    udp_pair = [safe_pair(a, b) for a, b in zip(u_s, u_d)]

    use_tcp = ~(df['tcp.srcport'].isna() & df['tcp.dstport'].isna())
    df['port_pair'] = [t if use else u for t, u, use in zip(tcp_pair, udp_pair, use_tcp.to_numpy())]
    df['ip_pair']   = [safe_pair(a, b) for a, b in zip(df['ip.src'], df['ip.dst'])]
    return df

def step_05_sessions(df: pd.DataFrame, session_timeout_s: int = 30) -> pd.DataFrame:
    """
    Compute session_id per (label, ip_pair, port_pair).
    Handles NaNs safely to avoid IntCastingNaNError.
    """
    df = df.copy()
    df = df.sort_values(['label', 'ip_pair', 'port_pair', 'ts'])

    # time diff per flow
    df['time_diff'] = (
        df.groupby(['label','ip_pair','port_pair'], dropna=False)['ts']
          .diff()
          .dt.total_seconds()
    )
    df['time_diff'] = df['time_diff'].fillna(0.0).clip(lower=0.0)

    # new session flag
    df['new_session'] = (df['time_diff'] > session_timeout_s).astype(np.int8)

    # session_id = cumulative sum of new_session within each flow
    # Use groupby-transform to align with original index, then fill and cast.
    session_cum = (
        df.groupby(['label','ip_pair','port_pair'], dropna=False)['new_session']
          .cumsum()
    )
    df['session_id'] = session_cum.fillna(0).astype('int64')

    return df

def step_06_session_aggregates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add packets_per_session and bytes_per_session by merging per-session aggregates.
    """
    df = df.copy()
    per_session = (
        df.groupby(['label','session_id'], dropna=False)
          .agg(packets_per_session=('frame.len','size'),
               bytes_per_session=('frame.len','sum'))
          .reset_index()
    )
    df = df.merge(per_session, on=['label','session_id'], how='left')
    return df

def step_07_directionality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Estimate outbound vs inbound per device; mark is_outbound.
    """
    df = df.copy()
    # Most frequent ip.src per label as device's primary source IP
    src_by_label = (
        df.groupby('label')['ip.src']
          .agg(lambda s: s.value_counts().index[0])
          .to_dict()
    )
    df['is_outbound'] = (df['ip.src'] == df['label'].map(src_by_label)).astype(np.int8)
    return df

# ================================
# 3) Orchestrator (run step by step)
# ================================


In [15]:
win_seconds=60
session_timeout_s=30
save_intermediate=False
basepath='.'

In [23]:
print("[step_00] prepare")
df0 = step_00_prepare(df)
if save_intermediate: df0.to_csv(f"{basepath}/_00_prepared.csv", index=False)

[step_00] prepare


In [24]:
print("[step_01] window bins")
df1 = step_01_window_bin(df0, win_seconds)
if save_intermediate: df1.to_csv(f"{basepath}/_01_timebin.csv", index=False)

[step_01] window bins


In [25]:
print("[step_02] inter-packet time")
df2 = step_02_inter_packet_time(df1)
if save_intermediate: df2.to_csv(f"{basepath}/_02_interpkt.csv", index=False)

[step_02] inter-packet time


In [26]:
print("[step_03] ports defaults")
df3 = step_03_ports_defaults(df2)
if save_intermediate: df3.to_csv(f"{basepath}/_03_ports.csv", index=False)

[step_03] ports defaults


In [34]:
print("[step_04] pairs")
df4 = step_04_pairs(df3)
if save_intermediate: df4.to_csv(f"{basepath}/_04_pairs.csv", index=False)

[step_04] pairs


In [35]:
print("[step_05] sessions")
df5 = step_05_sessions(df4, session_timeout_s)
if save_intermediate: df5.to_csv(f"{basepath}/_05_sessions.csv", index=False)

[step_05] sessions


In [ ]:
print("[step_06] session aggregates")
df6 = step_06_session_aggregates(df5)
if save_intermediate: df6.to_csv(f"{basepath}/_06_sess_aggs.csv", index=False)

[step_06] session aggregates


In [ ]:
print("[step_07] directionality")
df7 = step_07_directionality(df6)
if save_intermediate: df7.to_csv(f"{basepath}/_07_direction.csv", index=False)

In [ ]:
perwin.to_csv("CIC/engineered_features_combined_window_level.csv", index=False)
# perwin_norm.to_csv("CIC/engineered_features_combined_window_level_normalized.csv", index=False)

## Completed Feature Engineering

In [57]:
df.to_csv(f'{dataset}/engineered_dataset.csv', index=False)